# SQL with DuckDB

In this notebook, you will query local Parquet files with DuckDB and practice grouped summaries with SQL. The goal is to show that you can work directly from files without setting up a database server first.

**Learning goals:**
- Query Parquet files directly with DuckDB.
- Use SQL for summaries without loading everything into a warehouse.
- Compare small SQL questions with the batch transformations from the Polars notebook.

Before you run this notebook, make sure `data/raw/` contains local taxi Parquet files downloaded with `src/data_ingestion.py`.


## Outline

1. Connect to DuckDB.
2. Read Parquet files with SQL.
3. Build grouped summaries.
4. Write a few more practice queries.


In [ ]:
from pathlib import Path

import duckdb
import polars as pl

DATA_GLOB = "../data/raw/yellow_tripdata_2025-*.parquet"
if not list(Path("../data/raw").glob("yellow_tripdata_2025-*.parquet")):
    raise FileNotFoundError("Download local data first with src/data_ingestion.py.")

con = duckdb.connect()
con.sql("SELECT 1 AS ready").show()

## Step 1 - Query Parquet files directly

DuckDB can read Parquet files without an import step. That keeps the workflow lightweight and very close to the raw data.


In [ ]:
preview_query = """
SELECT
    VendorID,
    tpep_pickup_datetime,
    trip_distance,
    total_amount
FROM read_parquet(?)
LIMIT 5
"""

pl.from_arrow(con.execute(preview_query, [DATA_GLOB]).to_arrow_table())

## Step 2 - Build a summary with SQL

This query groups rides by vendor and month. The same pattern appears again in the final reporting notebook and script.


In [ ]:
monthly_vendor_query = """
SELECT
    VendorID,
    date_trunc('month', CAST(tpep_pickup_datetime AS TIMESTAMP)) AS revenue_month,
    COUNT(*) AS rides,
    ROUND(AVG(total_amount), 2) AS avg_total_amount,
    ROUND(SUM(total_amount), 2) AS total_revenue
FROM read_parquet(?)
GROUP BY 1, 2
ORDER BY 2, 1
"""

pl.from_arrow(con.execute(monthly_vendor_query, [DATA_GLOB]).to_arrow_table()).head(10)

## Step 3 - Ask a slightly different question

SQL is especially handy when you want to change the grouping without rewriting a whole DataFrame pipeline.


In [ ]:
payment_type_query = """
SELECT
    payment_type,
    COUNT(*) AS rides,
    ROUND(AVG(trip_distance), 2) AS avg_trip_distance,
    ROUND(AVG(total_amount), 2) AS avg_total_amount
FROM read_parquet(?)
GROUP BY 1
ORDER BY 1
"""

pl.from_arrow(con.execute(payment_type_query, [DATA_GLOB]).to_arrow_table())

## Exercises

- Limit the summary to rides with `trip_distance > 3`.
- Show the top five pickup zones by total revenue.
- Explain when SQL feels clearer than a DataFrame pipeline.


> **Common pitfall:** DuckDB can read a glob pattern, but the path still has to match real files on disk.


In [ ]:
# TODO: Write your solutions for the exercises above.


<details>
    <summary>Show Reference Solution</summary>

```python
trip_distance_query = """
SELECT
    payment_type,
    COUNT(*) AS rides,
    ROUND(AVG(trip_distance), 2) AS avg_trip_distance,
    ROUND(AVG(total_amount), 2) AS avg_total_amount
FROM read_parquet(?)
WHERE trip_distance > 3
GROUP BY 1
ORDER BY 1
"""

pl.from_arrow(con.execute(trip_distance_query, [DATA_GLOB]).to_arrow_table())

# ----------

top_pickup_query = """
SELECT
    PULocationID,
    ROUND(SUM(total_amount), 2) AS total_revenue
FROM read_parquet(?)
GROUP BY 1
ORDER BY total_revenue DESC
LIMIT 5
"""

pl.from_arrow(con.execute(top_pickup_query, [DATA_GLOB]).to_arrow_table())
```
</details>
